# Task 2 第三轮：ResNet18 与 ResNet34
本轮只新增 ResNet34，复用此前五组结果。研究问题：在统一训练条件下，更深模型的分类收益是否值得额外计算成本？

两组均采用各自的 ImageNet 预训练权重，输入 224、batch=128、AdamW 学习率 3e-4、weight_decay=1e-4、Dropout=0、标签平滑=0，训练 25 轮。划分、增强和采样规则不变。
ResNet34 的六类线性分类头按旧基线初始化流程重建，并记录与历史哈希是否一致；不一致时明确提示；骨干结构与预训练权重不同。这是预训练模型方案的比较，不能把全部差异归因于深度，也不代表各自充分调参后的上限。

**Kaggle 操作**：开启 GPU 和 Internet，添加比赛数据及唯一的第二轮完整备份 `task2-ablation-backup-round2`。从头运行，自动将已有五组恢复到 `/kaggle/working/task2_ablation_seed42`，仅训练 `resnet34`。
旧结果不完整或配置冲突时会在训练前停止。不要为绕过检查而删除旧配置。不要单独降低新组 batch 大小后仍称为统一条件比较。
已完成组自动跳过；未完成组再次运行会从第 1 轮重跑。本轮结束后运行最后的备份单元格，下载第三轮完整备份。

主指标仍为原图验证准确率，同时分析宏平均 F1、类别召回率、硬标签交叉熵、泛化差距、参数量和耗时。TTA 单列。旧实验没有记录硬件，跨会话耗时只能作为参考，不直接归因于模型结构。


In [ ]:
import hashlib
import json
import time
import random
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

SEED = 42
SPLIT_SEED = 42
PRETRAINED = True
EPOCHS = 25
BATCH_SIZE = 128       # 与旧基线一致；显存不足先停止，不单独改变新组批大小
IMAGE_SIZE = 224
LR = 3e-4
WORKERS = 0           # 本轮各组固定相同的数据加载方式
DATA_DIR = None        # 自动识别；存在多份数据时填写含 train/ 和 test/ 的目录

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.set_num_threads(4)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROOT = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path.cwd()
OUT = (Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()) / f"task2_ablation_seed{SEED}"
OUT.mkdir(parents=True, exist_ok=True)
CLASSES = ["Angry", "Fear", "Happy", "Neutral", "Sad", "Surprise"]
print("设备:", device, "| 输出目录:", OUT)

EXPERIMENTS = {
    "baseline":     {"weight_decay": 1e-4, "dropout": 0.0},
    "weight_decay": {"weight_decay": 1e-3, "dropout": 0.0},
    "dropout":      {"weight_decay": 1e-4, "dropout": 0.3},
    "combined":     {"weight_decay": 1e-3, "dropout": 0.3},
    "label_smoothing": {"weight_decay": 1e-4, "dropout": 0.0, "label_smoothing": 0.1},
    "resnet34": {"weight_decay": 1e-4, "dropout": 0.0, "architecture": "resnet34"},
}
RUN_EXPERIMENTS = ["resnet34"]  # 本轮只训练新增组，旧结果自动读取
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
# 改配置时换输出目录。已完成且配置一致的实验会跳过；未完成的实验从头重跑。


## 1. 读取数据并划分验证集


In [ ]:
if DATA_DIR is None:
    candidates = [p.parent for p in ROOT.rglob("train") if p.is_dir()
                  and (p.parent / "test").is_dir()
                  and sorted(d.name for d in p.iterdir() if d.is_dir()) == CLASSES]
    if len(candidates) != 1:
        raise ValueError(f"找到 {len(candidates)} 份数据，请设置 DATA_DIR。")
    DATA_DIR = candidates[0]
DATA_DIR = Path(DATA_DIR)

paths, labels, groups = [], [], defaultdict(list)
fingerprints = []
for label, name in enumerate(CLASSES):
    for path in sorted((DATA_DIR / "train" / name).glob("*.jpg")):
        with Image.open(path) as image:
            gray = image.convert("L")
            key = hashlib.sha256(str(gray.size).encode() + gray.tobytes()).hexdigest()
        fingerprints.append(key)
        groups[key].append(len(paths))
        paths.append(path)
        labels.append(label)
labels = np.array(labels)
totals = np.bincount(labels, minlength=len(CLASSES))
if (totals == 0).any():
    raise ValueError("训练数据缺少类别，请检查 DATA_DIR。")

# 按重复图片分组，平衡五份数据的类别比例，取第 0 份验证。
rng = np.random.default_rng(SPLIT_SEED)
keys = list(groups)
rng.shuffle(keys)
keys.sort(key=lambda key: -len(groups[key]))
folds = np.zeros(len(paths), dtype=int)
counts = np.zeros((5, len(CLASSES)), dtype=int)
for key in keys:
    indices = groups[key]
    histogram = np.bincount(labels[indices], minlength=len(CLASSES))
    costs = ((2 * counts * histogram + histogram ** 2) / totals).sum(axis=1)
    fold = int(rng.choice(np.flatnonzero(np.isclose(costs, costs.min(), rtol=0, atol=1e-12))))
    counts[fold] += histogram
    folds[indices] = fold
train_idx = np.flatnonzero(folds != 0)
val_idx = np.flatnonzero(folds == 0)
if (counts == 0).any():
    raise ValueError("部分划分缺少类别，请检查数据。")

test_ids = sorted(p.name for p in (DATA_DIR / "test").glob("*.jpg"))
# 优先沿用提交模板的 ID 顺序。
template = None
for folder in [DATA_DIR, *list(DATA_DIR.parents)[:3]]:
    for name in ["sample_submission.csv", "submission.csv"]:
        if (folder / name).is_file():
            frame = pd.read_csv(folder / name, dtype={"ID": str})
            if (list(frame.columns) == ["ID", "Emotion"] and len(frame) == len(test_ids)
                    and set(frame.ID) == set(test_ids)):
                template = frame
                break
    if template is not None:
        test_ids = template.ID.tolist()
        break
if not test_ids:
    raise ValueError("测试集为空。")

print(pd.Series(totals, index=CLASSES, name="图片数").to_string())
print(f"训练 {len(train_idx)} | 验证 {len(val_idx)} | 测试 {len(test_ids)}")
print("已分组的重复图片:", len(paths) - len(groups))

split = pd.DataFrame({"path": [p.relative_to(DATA_DIR).as_posix() for p in paths],
                      "class": [CLASSES[i] for i in labels], "pixel_sha256": fingerprints, "fold": folds})
SPLIT_HASH = hashlib.sha256(split.to_csv(index=False).encode()).hexdigest()
if (OUT / "split.csv").exists():
    old_hash = hashlib.sha256(pd.read_csv(OUT / "split.csv").to_csv(index=False).encode()).hexdigest()
    if old_hash != SPLIT_HASH:
        raise ValueError("此目录的数据划分已改变，请换输出目录。")
split.to_csv(OUT / "split.csv", index=False)


## 2. 数据增强与加载


In [ ]:
normalize = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(10, translate=(0.05, 0.05), scale=(0.95, 1.05),
                            interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(), normalize,
])
eval_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(), normalize,
])

class FaceDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths, self.labels, self.transform = paths, labels, transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            image = image.convert("RGB")
        return self.transform(image), int(self.labels[index])


def seed_worker(_):
    seed = torch.initial_seed() % (2 ** 32)
    random.seed(seed)
    np.random.seed(seed)


def loader(paths, labels, transform, shuffle=False):
    return DataLoader(FaceDataset(paths, labels, transform), batch_size=BATCH_SIZE,
                      shuffle=shuffle, num_workers=WORKERS, pin_memory=device.type == "cuda",
                      persistent_workers=WORKERS > 0, worker_init_fn=seed_worker,
                      generator=torch.Generator().manual_seed(SEED))


val_loader = loader([paths[i] for i in val_idx], labels[val_idx], eval_tf)


def progress(data_loader, stage):
    started = time.monotonic()
    total = len(data_loader)
    print(f"{stage}开始，共 {total} 批", flush=True)
    for step, batch in enumerate(data_loader, 1):
        yield batch
        if step == 1 or step % 20 == 0 or step == total:
            elapsed = time.monotonic() - started
            print(f"  {stage} {step}/{total} 批 | 已用 {elapsed:.0f}s", flush=True)


@torch.inference_mode()
def predict(model, data_loader, flip=False, stage="预测"):
    model.eval()
    base, tta = [], []
    for x, _ in progress(data_loader, stage):
        x = x.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
            logits = model(x)
            flipped = model(x.flip(-1)) if flip else None
        prob = logits.float().softmax(1)
        base.append(prob.cpu().numpy())
        tta.append(((prob + flipped.float().softmax(1)) / 2 if flip else prob).cpu().numpy())
    return np.concatenate(base), np.concatenate(tta)




## 3. 复用已完成结果，只训练新增组
先检查基线和已有实验的配置，再启动新训练。配置不一致时停止，避免将不同条件混作消融。
每组独立保存配置、训练历史和最佳模型；控制台不刷新时可查看 `history.csv`。
每组固定 25 轮，不提前停止；每轮固定随机种子及采样顺序。不同硬件仍可能产生数值差异。
完成后用最佳模型在 eval 模式、无增强训练集上评估，并计算训练/验证准确率差距。
已完成组自动跳过；未完成组再次运行会从头训练，不支持轮次级断点续训。


In [ ]:
def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def model_factory(architecture):
    if architecture == "resnet18":
        return models.resnet18, models.ResNet18_Weights.DEFAULT
    if architecture == "resnet34":
        return models.resnet34, models.ResNet34_Weights.DEFAULT
    raise ValueError(f"不支持的模型：{architecture}")


def build_model(dropout, pretrained=False, architecture="resnet18"):
    factory, weights = model_factory(architecture)
    model = factory(weights=weights if pretrained else None)
    model.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(model.fc.in_features, len(CLASSES)))
    if architecture == "resnet34":
        # 两模型分类头维度相同。复现旧 ResNet18 的初始化，不读取已训练的头。
        with torch.random.fork_rng():
            torch.manual_seed(SEED)
            reference = models.resnet18(weights=None)
            head = nn.Linear(reference.fc.in_features, len(CLASSES))
            model.fc[1].load_state_dict(head.state_dict())
        del reference, head
    return model.to(device)


def parameter_count(architecture):
    factory, _ = model_factory(architecture)
    with torch.device("meta"):
        model = factory(weights=None)
        model.fc = nn.Linear(model.fc.in_features, len(CLASSES))
    return sum(p.numel() for p in model.parameters())


def evaluate_probabilities(prob, truth):
    confusion = np.zeros((len(CLASSES), len(CLASSES)), dtype=int)
    np.add.at(confusion, (truth, prob.argmax(1)), 1)
    tp = confusion.diagonal()
    f1 = 2 * tp / np.maximum(confusion.sum(0) + confusion.sum(1), 1)
    return {"accuracy": float((prob.argmax(1) == truth).mean()),
            "loss": float(-np.log(prob[np.arange(len(truth)), truth].clip(1e-12, 1)).mean()),
            "macro_f1": float(f1.mean())}, confusion


def run_experiment(name):
    spec = EXPERIMENTS[name]
    folder = OUT / name
    folder.mkdir(parents=True, exist_ok=True)
    config = dict(experiment=name, **spec, seed=SEED, split_seed=SPLIT_SEED,
                  split_hash=SPLIT_HASH, epochs=EPOCHS, batch_size=BATCH_SIZE,
                  image_size=IMAGE_SIZE, lr=LR, workers=WORKERS, pretrained=PRETRAINED,
                  classes=CLASSES, model=spec.get("architecture", "resnet18"),
                  pipeline="architecture_v1" if "architecture" in spec else ("label_smoothing_v1" if "label_smoothing" in spec else "wd_dropout_v1"),
                  torch_version=str(torch.__version__), torchvision_version=__import__("torchvision").__version__)
    if (folder / "config.json").exists():
        previous = json.loads((folder / "config.json").read_text())
        if previous != config:
            raise ValueError(f"{name} 的配置已改变，请换 OUT 或新的实验名称。")
    else:
        (folder / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
    if (folder / "metrics.json").exists() and (folder / "best.pt").exists():
        print(f"{name} 已完成，读取已保存结果。", flush=True)
        saved = json.loads((folder / "metrics.json").read_text())
        saved.setdefault("label_smoothing", 0.0)  # 兼容第一轮文件，不改写旧结果
        saved.setdefault("model", config["model"])
        saved.setdefault("parameters", parameter_count(config["model"]))
        saved.setdefault("device_name", "未记录")
        return saved

    seed_all(SEED)
    model = build_model(spec["dropout"], pretrained=PRETRAINED, architecture=config["model"])
    head_hash = hashlib.sha256(b"".join(t.detach().cpu().numpy().tobytes()
                                      for t in model.fc.state_dict().values())).hexdigest()
    if name == "resnet34":
        baseline_metrics = json.loads((OUT / "baseline" / "metrics.json").read_text())
        head_matches_baseline = head_hash == baseline_metrics["initial_head_hash"]
        print("与旧基线分类头初始化哈希一致：", head_matches_baseline, flush=True)
        if not head_matches_baseline:
            print("当前环境未精确复现旧分类头，将记录此差异；不能声称初始权重完全相同。", flush=True)
    print(f"模型 {config['model']}，参数量 {sum(p.numel() for p in model.parameters()):,}", flush=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=spec["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, EPOCHS)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    criterion = nn.CrossEntropyLoss(label_smoothing=spec.get("label_smoothing", 0.0))
    train_loader = loader([paths[i] for i in train_idx], labels[train_idx], train_tf, True)
    best, history = -1.0, []
    started = time.monotonic()
    for epoch in range(1, EPOCHS + 1):
        seed_all(SEED + epoch)
        train_loader.generator.manual_seed(SEED + epoch)
        model.train()
        loss_sum, nll_sum, correct, total = 0.0, 0.0, 0, 0
        for x, y in progress(train_loader, f"{name} Epoch {epoch:02d} 训练"):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
                logits = model(x)
                loss = criterion(logits, y)
            if not torch.isfinite(loss):
                raise RuntimeError("训练损失异常。")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.item() * len(y)
            # 只记录，不参与反传；与旧组普通交叉熵采用相同定义。
            nll_sum += nn.functional.cross_entropy(logits.detach().float(), y).item() * len(y)
            correct += (logits.argmax(1) == y).sum().item()
            total += len(y)
        val_prob, _ = predict(model, val_loader, stage=f"{name} Epoch {epoch:02d} 验证")
        metrics, _ = evaluate_probabilities(val_prob, labels[val_idx])
        history.append(dict(epoch=epoch, train_loss=loss_sum / total, train_nll=nll_sum / total,
                            train_accuracy=correct / total,
                            val_accuracy=metrics["accuracy"], val_loss=metrics["loss"],
                            lr=optimizer.param_groups[0]["lr"]))
        pd.DataFrame(history).to_csv(folder / "history.csv", index=False)
        print(f"{name} Epoch {epoch:02d}: train_acc={correct/total:.4f} "
              f"val_acc={metrics['accuracy']:.4f} val_loss={metrics['loss']:.4f}", flush=True)
        if metrics["accuracy"] > best:
            best = metrics["accuracy"]
            torch.save({"model": model.state_dict(), "config": config,
                        "epoch": epoch, "accuracy": best}, folder / "best.tmp")
            (folder / "best.tmp").replace(folder / "best.pt")
        scheduler.step()
    checkpoint = torch.load(folder / "best.pt", map_location="cpu", weights_only=True)
    model.load_state_dict(checkpoint["model"])
    val_base, val_tta = predict(model, val_loader, flip=True, stage=f"{name} 最佳模型验证")
    metrics, confusion = evaluate_probabilities(val_base, labels[val_idx])
    train_eval_loader = loader([paths[i] for i in train_idx], labels[train_idx], eval_tf)
    train_prob, _ = predict(model, train_eval_loader, stage=f"{name} 无增强训练集评估")
    train_acc = float((train_prob.argmax(1) == labels[train_idx]).mean())
    recall = confusion.diagonal() / np.maximum(confusion.sum(1), 1)
    result = dict(experiment=name, **spec, best_epoch=checkpoint["epoch"],
                  val_accuracy=metrics["accuracy"], val_loss=metrics["loss"], macro_f1=metrics["macro_f1"],
                  tta_accuracy=float((val_tta.argmax(1) == labels[val_idx]).mean()),
                  train_eval_accuracy=train_acc, gap_pp=100 * (train_acc - metrics["accuracy"]),
                  minutes=(time.monotonic() - started) / 60, seed=SEED, initial_head_hash=head_hash,
                  **{f"recall_{c}": float(r) for c, r in zip(CLASSES, recall)})
    result.update(model=config["model"], parameters=sum(p.numel() for p in model.parameters()),
                  device_name=torch.cuda.get_device_name(device) if device.type == "cuda" else "CPU")
    if name == "resnet34":
        result["head_matches_baseline"] = head_matches_baseline
    result.setdefault("label_smoothing", 0.0)
    pd.DataFrame(confusion, index=CLASSES, columns=CLASSES).to_csv(folder / "confusion_matrix.csv")
    validation = pd.DataFrame(val_base, columns=CLASSES)
    validation.insert(0, "true_class", [CLASSES[i] for i in labels[val_idx]])
    validation.insert(0, "path", [paths[i].relative_to(DATA_DIR).as_posix() for i in val_idx])
    validation.to_csv(folder / "validation_probabilities.csv", index=False)
    (folder / "metrics.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
    del model, optimizer, train_loader, train_eval_loader, checkpoint
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return result


def restore_previous_results(input_root=Path("/kaggle/input")):
    import shutil
    old_names = list(EXPERIMENTS)  # 第三轮备份若包含 ResNet34，也一并恢复，避免重跑
    required = ["config.json", "metrics.json", "best.pt", "history.csv"]
    if all((OUT / name / f).exists() for name in old_names for f in required):
        return
    candidates = [p.parent for p in input_root.rglob("split.csv")
                  if all((p.parent / "baseline" / f).is_file() for f in required)]
    if not candidates:
        return  # 后续完整性检查给出缺失文件。
    if len(candidates) != 1:
        raise RuntimeError("发现多份实验备份，请在 Input 中仅保留本轮要恢复的一份。")
    src = candidates[0]
    if not pd.read_csv(src / "split.csv").equals(pd.read_csv(OUT / "split.csv")):
        raise ValueError("备份划分与当前划分不同，停止恢复。")
    pending = []
    for name in old_names:
        for source in (src / name).rglob("*"):
            if not source.is_file():
                continue
            target = OUT / source.relative_to(src)
            if target.exists():
                if hashlib.sha256(source.read_bytes()).digest() != hashlib.sha256(target.read_bytes()).digest():
                    raise ValueError(f"已有文件与备份冲突，不覆盖：{target}")
            else:
                pending.append((source, target))
    # 所有冲突检查通过后，只复制缺失文件，保留已有训练结果。
    for source, target in pending:
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, target)
    print(f"已从 {src} 恢复 {len(pending)} 个文件到 {OUT}", flush=True)


def run_selected_experiments():
    unknown = set(RUN_EXPERIMENTS) - set(EXPERIMENTS)
    if unknown:
        raise ValueError(f"未知实验：{unknown}")
    if {"label_smoothing", "resnet34"}.intersection(RUN_EXPERIMENTS):
        restore_previous_results()
        required_names = ["baseline", "weight_decay", "dropout", "combined", "label_smoothing"] if "resnet34" in RUN_EXPERIMENTS else ["baseline"]
        required = [OUT / "split.csv"] + [OUT / name / filename
                    for name in required_names
                    for filename in ["config.json", "metrics.json", "best.pt", "history.csv"]]
        missing = [str(p) for p in required if not p.exists()]
        if missing:
            raise FileNotFoundError("请添加第二轮完整备份并恢复到 OUT，不会自动重跑旧实验。缺少：\n" + "\n".join(missing))
    collected = {}
    # 先校验并复用所有已完成组，避免训练一小时后才发现旧配置冲突。
    for name in EXPERIMENTS:
        if (OUT / name / "metrics.json").exists() and (OUT / name / "best.pt").exists():
            if not (OUT / name / "config.json").exists():
                raise FileNotFoundError(f"{name} 缺少 config.json，请恢复完整备份。")
            collected[name] = run_experiment(name)
    for name in RUN_EXPERIMENTS:
        if name not in collected:
            collected[name] = run_experiment(name)
        pd.DataFrame(collected.values()).to_csv(OUT / "comparison.csv", index=False)
    ordered = [collected[name] for name in EXPERIMENTS if name in collected]
    pd.DataFrame(ordered).to_csv(OUT / "comparison.csv", index=False)
    return ordered


results = run_selected_experiments()
print("所选实验已完成，结果目录:", OUT)


## 4. 对照分析
主要比较原图准确率、宏平均 F1、各类召回率及验证损失。`gap_pp` 为最佳模型在无增强、eval 模式下的训练/验证准确率差，单位是百分点。TTA 单列，避免混淆贡献。

标签平滑改变训练目标，不能直接用平滑后的 `train_loss` 与旧组比较；曲线统一展示硬标签 `train_nll`（旧组等于 `train_loss`）。验证损失始终采用同一定义。


In [ ]:
comparison = pd.DataFrame(results).set_index("experiment")
if comparison.loc[comparison.model == "resnet18", "initial_head_hash"].nunique() != 1:
    raise RuntimeError("ResNet18 消融组分类头初始权重不一致，请检查随机种子设置。")
columns = ["model", "parameters", "weight_decay", "dropout", "label_smoothing", "best_epoch", "val_accuracy", "tta_accuracy",
           "macro_f1", "val_loss", "train_eval_accuracy", "gap_pp", "minutes"]
display(comparison[columns].round(4))
display(comparison[[f"recall_{c}" for c in CLASSES]].round(4))

if {"baseline", "weight_decay", "dropout", "combined"}.issubset(comparison.index):
    a, b, c, d = (comparison.loc[name, "val_accuracy"]
                  for name in ["baseline", "weight_decay", "dropout", "combined"])
    effects = pd.DataFrame({"effect": ["weight_decay - baseline", "dropout - baseline",
                                       "combined - baseline", "combined - weight_decay - dropout + baseline"],
                            "change_pp": 100 * np.array([b-a, c-a, d-a, d-b-c+a])})
    effects.to_csv(OUT / "ablation_effects.csv", index=False)
    display(effects)
else:
    print("四组尚未齐全，暂不计算完整消融差值。")

# 第二轮成对比较：准确率/F1/召回率用百分点，其余保持原单位。
if {"baseline", "label_smoothing"}.issubset(comparison.index):
    paired_metrics = ["val_accuracy", "tta_accuracy", "macro_f1", "val_loss", "gap_pp"]
    paired_metrics += [f"recall_{c}" for c in CLASSES]
    paired = comparison.loc[["baseline", "label_smoothing"], paired_metrics].T.copy()
    factors = np.array([100 if m.endswith("accuracy") or m == "macro_f1" or m.startswith("recall_")
                        else 1 for m in paired.index])
    paired["change"] = (paired.label_smoothing - paired.baseline) * factors
    paired["change_unit"] = ["pp" if f == 100 or m == "gap_pp" else "NLL"
                             for m, f in zip(paired.index, factors)]
    paired.to_csv(OUT / "label_smoothing_comparison.csv", index_label="metric")
    print("第二轮：change = label_smoothing − baseline；准确率/F1/召回率越高越好，损失越低越好。")
    display(paired.round(4))

# 第三轮：仅与未加额外正则化的旧基线比较模型结构方案。
if {"baseline", "resnet34"}.issubset(comparison.index):
    fields = ["parameters", "val_accuracy", "tta_accuracy", "macro_f1", "val_loss", "gap_pp", "minutes"]
    fields += [f"recall_{c}" for c in CLASSES]
    architecture_pair = comparison.loc[["baseline", "resnet34"], fields].T.copy()
    factors = np.array([100 if f.endswith("accuracy") or f == "macro_f1" or f.startswith("recall_") else 1 for f in fields])
    architecture_pair["change"] = (architecture_pair.resnet34 - architecture_pair.baseline) * factors
    architecture_pair["change_unit"] = ["pp" if k == 100 or f == "gap_pp" else
                                          {"parameters": "parameters", "minutes": "minutes", "val_loss": "NLL"}[f]
                                          for f, k in zip(fields, factors)]
    architecture_pair.to_csv(OUT / "model_comparison.csv", index_label="metric")
    print("分类头初始化哈希与基线一致：", comparison.loc["resnet34", "initial_head_hash"] == comparison.loc["baseline", "initial_head_hash"])
    print("第三轮：ResNet34 − baseline；同训练预算、各自预训练权重的模型方案比较。")
    display(architecture_pair.round(4))
    display(comparison.loc[["baseline", "resnet34"], ["model", "device_name", "minutes"]])
    print("旧实验未记录设备；跨会话耗时仅供参考，不直接归因于结构。相同轮数不等于相同计算量。")

# 可用于报告的曲线。训练准确率来自带增强的 train 模式，不用于直接比较泛化差距。
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("曲线绘制需要 matplotlib；逐轮数据已保存在各组 history.csv。")
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for name in comparison.index:
        h = pd.read_csv(OUT / name / "history.csv")
        axes[0].plot(h.epoch, h["train_nll"] if "train_nll" in h else h.train_loss, label=name)
        axes[1].plot(h.epoch, h.val_accuracy, label=name)
        axes[2].plot(h.epoch, h.val_loss, label=name)
    for ax, title in zip(axes, ["Training NLL (hard labels)", "Validation accuracy", "Validation NLL (hard labels)"]):
        ax.set(xlabel="Epoch", title=title)
        ax.grid(alpha=0.2)
        ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "learning_curves.png", dpi=180)
    plt.show()
print("单个种子的结果是初步证据；微小差异需要重复实验确认。")


## 5. 用选出的最佳模型预测测试集
本轮只对验证集选出的模型执行测试集预测，避免把测试集用于超参数选择。


In [ ]:
# 先按原图验证准确率选实验，同分时选验证损失更低者。
winner = comparison.sort_values(["val_accuracy", "val_loss"], ascending=[False, True]).index[0]
checkpoint = torch.load(OUT / winner / "best.pt", map_location="cpu", weights_only=True)
model = build_model(checkpoint["config"]["dropout"], architecture=checkpoint["config"].get("model", "resnet18"))
model.load_state_dict(checkpoint["model"])
use_tta = comparison.loc[winner, "tta_accuracy"] > comparison.loc[winner, "val_accuracy"]
PRED_OUT = OUT / "predictions" / winner
PRED_OUT.mkdir(parents=True, exist_ok=True)
print(f"选用 {winner}，第 {checkpoint['epoch']} 轮，使用 TTA: {use_tta}", flush=True)
test_loader = loader([DATA_DIR / "test" / name for name in test_ids],
                     np.zeros(len(test_ids), dtype=int), eval_tf)
test_base, test_tta = predict(model, test_loader, flip=use_tta, stage="测试集预测")
test_prob = test_tta if use_tta else test_base
if not np.isfinite(test_prob).all() or not np.allclose(test_prob.sum(1), 1, atol=1e-5):
    raise RuntimeError("预测概率异常。")
probabilities = pd.DataFrame(test_prob, columns=CLASSES)
probabilities.insert(0, "ID", test_ids)
probabilities.to_csv(PRED_OUT / "test_probabilities.csv", index=False)
predictions = pd.DataFrame({"ID": test_ids, "EmotionName": [CLASSES[i] for i in test_prob.argmax(1)]})
predictions.to_csv(PRED_OUT / "predictions_labels.csv", index=False)
(PRED_OUT / "selection.json").write_text(json.dumps(dict(experiment=winner, epoch=checkpoint["epoch"],
                                                        tta=bool(use_tta)), indent=2), encoding="utf-8")
display(predictions.head())
print("预测已保存:", PRED_OUT)


## 6. 导出官方数字标签提交文件
比赛说明给出的官方编码：Angry=0、Fear=1、Happy=2、Sad=3、Surprise=4、Neutral=5。训练内部类别顺序保持不变，仅在导出时按名称映射。输出 `submission_official.csv`，保留之前的试验提交文件；无需重训或重新预测。


In [ ]:
from IPython.display import FileLink, display

LABEL_MAP = {"Angry": 0, "Fear": 1, "Happy": 2, "Sad": 3, "Surprise": 4, "Neutral": 5}

predictions = pd.read_csv(PRED_OUT / "predictions_labels.csv", dtype={"ID": str})
submission = predictions.rename(columns={"EmotionName": "Emotion"})[["ID", "Emotion"]].copy()
if (submission.isna().any().any() or submission.ID.duplicated().any()
        or not submission.Emotion.isin(CLASSES).all()):
    raise ValueError("提交结果包含空值、重复 ID 或未知类别。")
if len(submission) != len(test_ids) or set(submission.ID) != set(test_ids):
    raise ValueError("提交 ID 与测试集不一致。")
submission = submission.set_index("ID").loc[test_ids].reset_index()
submission["Emotion"] = submission["Emotion"].map(LABEL_MAP)
if submission.Emotion.isna().any():
    raise ValueError("存在未映射的类别，请检查预测名称。")
submission["Emotion"] = submission["Emotion"].astype("int64")
submission_path = PRED_OUT / "submission_official.csv"
submission.to_csv(submission_path, index=False)
display(submission.head())
print("提交文件:", submission_path, "| 行数:", len(submission))
display(FileLink(str(submission_path.relative_to(Path.cwd()))
                 if submission_path.is_relative_to(Path.cwd()) else str(submission_path)))


## 报告分析要点
- 权重衰减与 Dropout 分别是否改善验证表现？组合是否有额外收益？即使效果下降也保留结果。
- 训练准确率下降不等于泛化改善；结合验证损失、同条件泛化差距和各类召回率解释。
- 1e-3 与 0.3 是本轮候选，不能称为最优超参数。后续根据结果选择范围，并用多个种子检查稳定性。
- 此验证集用于选择轮次和配置，不能将结果表述为独立测试集成绩。
- 实验记录见各组 config.json、history.csv、metrics.json；根目录含 split.csv、comparison.csv、ablation_effects.csv 和可选曲线图。

参考：[AdamW 论文](https://arxiv.org/abs/1711.05101)、[PyTorch Dropout 文档](https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html)。

第二轮报告要点：
- 动机：第一轮验证准确率缓慢上升，但验证交叉熵明显回升，提出“预测过度自信”的待验证解释。
- 控制变量：只将标签平滑从 0 改为 0.1，其他设置保持与 baseline 一致。
- 同时看原图准确率、硬标签验证交叉熵、宏平均 F1、类别召回率和泛化差距。损失降低但准确率不升，只能说明对应损失改善。
- `label_smoothing_comparison.csv` 保存成对差值；正结果与负结果都记录。单个候选、单个种子不能证明最优或稳定提升。
- 参考：[When Does Label Smoothing Help?](https://arxiv.org/abs/1906.02629)。

第三轮报告要点：
- 对照问题：更深的预训练模型在同一训练条件下能否改善分类，是否值得增加参数和训练成本？
- 只新增 ResNet34，复用基线；不同时增加标签平滑、Dropout 或增强。
- 同种子不自动意味着相同分类头初始化；代码按基线流程重建分类头并核对哈希；环境差异下未能精确复现时如实记录，不能声称权重完全相同。
- 各自使用预训练权重，因此不能把结果严格解释为深度的独立因果效应；相同超参数也不代表每种模型均达到最佳。
- `model_comparison.csv` 保存成对对照。若收益很小，后续对两模型同时增加随机种子检查稳定性。


## 7. 保存第三轮完整备份
所有实验完成后运行，点击链接下载。用于下次恢复的备份需要包含全部六组结果。


In [ ]:
import shutil
from IPython.display import FileLink, display

assert (OUT / "resnet34" / "metrics.json").exists(), "ResNet34 尚未完成，请先检查训练状态"
archive_base = OUT.parent / "task2_ablation_backup_round3"
shutil.make_archive(str(archive_base), "zip", root_dir=OUT.parent, base_dir=OUT.name)
print("备份已生成，请下载后再关闭会话：")
display(FileLink(str(archive_base.with_suffix(".zip").relative_to(Path.cwd()))
                 if archive_base.with_suffix(".zip").is_relative_to(Path.cwd())
                 else str(archive_base.with_suffix(".zip"))))
